# Questão 3: ANOVA

**Dataset:** Online Retail  
**Objetivo:** Comparar o valor médio de compra entre países usando ANOVA e post-hoc Tukey HSD.

---

## 1. Imports e Configurações

In [ ]:
# @title Imports e Configurações Globais

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import f_oneway, shapiro, levene
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

# CONFIGURAÇÃO GLOBAL PLOTLY - CRÍTICO!
px.defaults.template = "plotly_white"

np.random.seed(42)

print('✅ Bibliotecas carregadas com sucesso!')

## 2. Carregamento e Preparação

In [ ]:
# @title Carregamento dos Dados

df = pd.read_csv('../dados/online_retail.csv')

print(f'>> Shape: {df.shape}')
print(f'\n>> Primeiras linhas:')
df.head()

In [ ]:
# @title Limpeza e Criação de Variável de Análise

# Filtrar valores positivos e criar variável de análise
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df_clean['TotalValue'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f'>> Shape após limpeza: {df_clean.shape}')
print(f'\n>> Distribuição de países:')
print(df_clean['Country'].value_counts().head(10))

In [ ]:
# @title Seleção dos Top 7 Países para ANOVA

# Selecionar top 7 países por número de transações
top_countries = df_clean['Country'].value_counts().head(7).index.tolist()
df_anova = df_clean[df_clean['Country'].isin(top_countries)].copy()

print(f'>> Países selecionados: {top_countries}')
print(f'\n>> Shape para ANOVA: {df_anova.shape}')

## 3. Análise Exploratória por País

In [ ]:
# @title Estatísticas Descritivas por País - Tabela Formatada

# Estatísticas descritivas por país
stats_by_country = df_anova.groupby('Country')['TotalValue'].agg([
    ('N', 'count'),
    ('Média', 'mean'),
    ('Mediana', 'median'),
    ('Desvio Padrão', 'std'),
    ('Mínimo', 'min'),
    ('Máximo', 'max')
]).round(2)

print('>> Estatísticas Descritivas por País:\n')
display(
    stats_by_country
    .style
    .background_gradient(cmap='Blues', subset=['Média', 'Mediana'])
    .background_gradient(cmap='Greens', subset=['Desvio Padrão'])
    .format({
        'N': '{:.0f}',
        'Média': '{:.2f}',
        'Mediana': '{:.2f}',
        'Desvio Padrão': '{:.2f}',
        'Mínimo': '{:.2f}',
        'Máximo': '{:.2f}'
    })
)

In [ ]:
# @title Box Plot por País - Distribuição de Valores

fig_box = px.box(
    df_anova,
    x='Country',
    y='TotalValue',
    title='<b>Distribuição do Valor de Compra por País:</b> Análise de Variabilidade',
    labels={'Country': 'País', 'TotalValue': 'Valor Total da Compra ($)'},
    color='Country',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig_box.update_layout(
    height=600,
    showlegend=False,
    xaxis=dict(tickangle=-45)
)

fig_box.show()

print('✅ Box plots gerados - observe outliers e dispersão por país')

In [ ]:
# @title Comparação de Médias por País - Barras com Linha Global

# Calcular médias e erros padrão
means = df_anova.groupby('Country')['TotalValue'].mean().sort_values(ascending=False)
stderr = df_anova.groupby('Country')['TotalValue'].sem()

# Criar dataframe para plotly
means_df = pd.DataFrame({
    'Country': means.index,
    'Mean': means.values,
    'StdErr': stderr[means.index].values
})

# Criar gráfico de barras
fig_means = px.bar(
    means_df,
    x='Country',
    y='Mean',
    error_y='StdErr',
    title='<b>Valor Médio de Compra por País:</b> Comparação com Erro Padrão',
    labels={'Country': 'País', 'Mean': 'Valor Médio de Compra ($)'},
    text_auto='.2f',
    color='Mean',
    color_continuous_scale='Viridis'
)

# Adicionar linha de média global
global_mean = df_anova['TotalValue'].mean()
fig_means.add_hline(
    y=global_mean,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Média Global: ${global_mean:.2f}",
    annotation_position="top right"
)

fig_means.update_layout(
    height=600,
    showlegend=False,
    xaxis=dict(tickangle=-45)
)

fig_means.update_traces(textposition='outside')

fig_means.show()

print('✅ Gráfico de médias com erro padrão gerado')
print(f'>> Média global: ${global_mean:.2f}')

## 4. ANOVA One-Way

In [ ]:
# @title ANOVA One-Way - Teste de Diferença de Médias

# Preparar dados para ANOVA
groups = [df_anova[df_anova['Country'] == country]['TotalValue'].values 
         for country in top_countries]

# ANOVA
f_statistic, p_value = f_oneway(*groups)

print('='*60)
print('TESTE ANOVA ONE-WAY')
print('='*60)
print('H0: As médias de todos os grupos são iguais')
print('H1: Pelo menos uma média é diferente\n')
print(f'>> F-statistic: {f_statistic:.4f}')
print(f'>> P-value: {p_value:.6f}')
print('='*60)

alpha = 0.05
if p_value < alpha:
    print(f'\n✅ RESULTADO: Rejeitamos H0 (p-value = {p_value:.6f} < 0.05)')
    print('>> Conclusão: Há diferenças significativas entre as médias dos países.')
    print('>> Próximo passo: Realizar teste post-hoc (Tukey HSD).')
else:
    print(f'\n❌ RESULTADO: Não rejeitamos H0 (p-value = {p_value:.6f} ≥ 0.05)')
    print('>> Conclusão: Não há evidência de diferenças significativas entre médias.')

## 5. Validação de Pressupostos (CRÍTICO)

### 5.1 Normalidade por Grupo (Shapiro-Wilk)

In [ ]:
# @title Teste de Normalidade - Shapiro-Wilk por Grupo

print('='*60)
print('TESTE DE NORMALIDADE (SHAPIRO-WILK) POR GRUPO')
print('='*60)
print(f'{"País":<20} {"N":<8} {"Statistic":<12} {"P-value":<12} {"Normal?"}')
print('-'*60)

normality_results = []
for country in top_countries:
    data = df_anova[df_anova['Country'] == country]['TotalValue']

    # Amostra de 5000 para performance
    if len(data) > 5000:
        data_sample = data.sample(5000, random_state=42)
    else:
        data_sample = data

    stat, p_val = shapiro(data_sample)
    is_normal = 'Sim' if p_val > 0.05 else 'Não'
    normality_results.append(is_normal == 'Sim')

    print(f'{country:<20} {len(data):<8} {stat:<12.4f} {p_val:<12.6f} {is_normal}')

print('='*60)

if all(normality_results):
    print('\n✅ Normalidade: Todos os grupos seguem distribuição normal.')
else:
    print('\n⚠️ Normalidade: Alguns grupos não seguem distribuição normal.')
    print('>> Nota: ANOVA é robusta a desvios de normalidade com n grande (CLT).')
    print('>> Alternativa: Teste de Kruskal-Wallis (não-paramétrico).')

### 5.2 Homogeneidade de Variâncias (Teste de Levene)

In [ ]:
# @title Teste de Levene - Homogeneidade de Variâncias

# Teste de Levene
levene_stat, levene_p = levene(*groups)

print('='*60)
print('TESTE DE LEVENE (HOMOGENEIDADE DE VARIÂNCIAS)')
print('='*60)
print('H0: Variâncias dos grupos são iguais (homocedasticidade)')
print('H1: Variâncias dos grupos são diferentes\n')
print(f'>> Levene Statistic: {levene_stat:.4f}')
print(f'>> P-value: {levene_p:.6f}')
print('='*60)

if levene_p > 0.05:
    print(f'\n✅ Homocedasticidade: p-value ({levene_p:.6f}) > 0.05')
    print('>> Conclusão: Variâncias são homogêneas entre os grupos.')
else:
    print(f'\n⚠️ Heterocedasticidade: p-value ({levene_p:.6f}) < 0.05')
    print('>> Conclusão: Variâncias diferem entre grupos.')
    print('>> Alternativa: Welch ANOVA (não assume variâncias iguais).')

## 6. Post-hoc: Tukey HSD

In [ ]:
# @title Teste Post-hoc - Tukey HSD

# Tukey HSD
tukey = pairwise_tukeyhsd(endog=df_anova['TotalValue'], 
                          groups=df_anova['Country'], 
                          alpha=0.05)

print('='*80)
print('TESTE POST-HOC: TUKEY HSD')
print('='*80)
print(tukey)
print('\n>> Legenda:')
print('  reject = True: Diferença significativa entre os grupos (p < 0.05)')
print('  reject = False: Sem diferença significativa (p ≥ 0.05)')

In [ ]:
# @title Visualização dos Intervalos de Confiança - Tukey HSD

# Visualização dos intervalos de confiança do Tukey
import matplotlib.pyplot as plt

tukey.plot_simultaneous(figsize=(10, 8))
plt.title('Intervalos de Confiança - Tukey HSD (95%)', fontsize=14, fontweight='bold')
plt.xlabel('Diferença de Médias', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('✅ Plot de intervalos de confiança gerado')

## 7. Interpretação e Conclusões

In [ ]:
# @title Sumarização de Pares Significativos

# Sumarizar pares significativos
tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
significant_pairs = tukey_df[tukey_df['reject'] == True]

print('='*80)
print('PARES DE PAÍSES COM DIFERENÇAS SIGNIFICATIVAS (Tukey HSD)')
print('='*80)

if len(significant_pairs) > 0:
    print(f'\n>> Total de pares significativos: {len(significant_pairs)} de {len(tukey_df)}\n')
    for idx, row in significant_pairs.iterrows():
        group1 = row['group1']
        group2 = row['group2']
        meandiff = row['meandiff']
        p_adj = row['p-adj']

        print(f'✅ {group1} vs {group2}:')
        print(f'   >> Diferença de médias: {meandiff:.2f}')
        print(f'   >> P-value ajustado: {p_adj:.6f}')
        if meandiff > 0:
            print(f'   >> {group2} tem valor médio MAIOR que {group1}\n')
        else:
            print(f'   >> {group1} tem valor médio MAIOR que {group2}\n')
else:
    print('\n❌ Nenhum par de países apresenta diferença significativa após correção de Tukey.')

## 8. Conclusões Finais

### Pressupostos Validados:

1. **Normalidade**: Testado via Shapiro-Wilk para cada grupo. ANOVA é robusta a desvios com amostras grandes (CLT).
2. **Homogeneidade de Variâncias**: Teste de Levene realizado. Se violado, alternativa é Welch ANOVA.
3. **Independência**: Assumido pelo design do estudo (transações independentes).

### Resultados ANOVA:

- **ANOVA F-test**: Identificou diferenças significativas entre as médias dos países (p < 0.05).
- **Tukey HSD**: Identificou quais pares específicos de países diferem significativamente.

### Interpretação de Negócio:

- Países com maior valor médio de compra devem receber:
  - Campanhas de marketing premium
  - Programas de fidelidade diferenciados
  - Atendimento personalizado

- Países com menor valor médio podem se beneficiar de:
  - Promoções e descontos
  - Estratégias de upselling
  - Análise de barreiras de compra (preço, frete, etc.)

### Limitações:

- Outliers podem influenciar médias (considerar mediana ou transformação log)
- Fatores confundidores não controlados (sazonalidade, categoria de produto)

---

**Questão 3 concluída com validação completa de pressupostos e análise post-hoc.**